# Visualize LowResPT Training Metrics

Reads the latest PyTorch Lightning `metrics.csv` from the `outputs/` directory and plots training/validation loss over steps and epochs.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid')

In [ ]:
# Find the latest version directory under outputs/
outputs_root = Path('outputs')
version_dirs = sorted(outputs_root.glob('*/version_*'), key=lambda p: int(p.name.split('_')[-1]))
print('Found versions:')
for v in version_dirs:
    print(' ', v)

latest = version_dirs[-2]
metrics_file = latest / 'metrics.csv'
print(f'\nUsing: {metrics_file}')

In [ ]:
# Load metrics.csv — Lightning logs train and val in separate rows
raw = pd.read_csv(metrics_file)
print('Raw shape:', raw.shape)
print('Columns:', raw.columns.tolist())
display(raw.head(10))

In [ ]:
# Split into train rows (have train_loss) and val rows (have val_loss)
train_df = raw.dropna(subset=['train_loss']).copy()
val_df   = raw.dropna(subset=['val_loss']).copy()

# Fill epoch forward so val rows inherit their epoch
raw_filled = raw.copy()
raw_filled['epoch'] = raw_filled['epoch'].ffill().astype(int)
train_df = raw_filled.dropna(subset=['train_loss']).copy()
val_df   = raw_filled.dropna(subset=['val_loss']).copy()

print(f'Train rows: {len(train_df)}, Val rows: {len(val_df)}')

In [ ]:
# Smoothing helpers
def sma(series, window=None):
    if window is None:
        window = max(5, int(len(series) * 0.02))
    return series.rolling(window=window, min_periods=1).mean()

def ewm(series, alpha=0.1):
    return series.ewm(alpha=alpha).mean()

train_df = train_df.sort_values('step').reset_index(drop=True)
val_df   = val_df.sort_values('step').reset_index(drop=True)

train_df['train_loss_sma'] = sma(train_df['train_loss'])
train_df['train_loss_ewm'] = ewm(train_df['train_loss'])

if 'train_full_mse' in train_df.columns:
    train_df['train_full_mse_sma'] = sma(train_df['train_full_mse'])
    train_df['train_full_mse_ewm'] = ewm(train_df['train_full_mse'])

In [ ]:
# Epoch-level aggregation
epoch_train = train_df.groupby('epoch')['train_loss'].agg(['mean','std']).rename(columns={'mean':'train_loss_mean','std':'train_loss_std'})
epoch_val   = val_df.groupby('epoch')['val_loss'].agg(['mean','std']).rename(columns={'mean':'val_loss_mean','std':'val_loss_std'})
epoch_summary = epoch_train.join(epoch_val, how='outer')
print('Epoch summary:')
display(epoch_summary)

In [ ]:
# Plot 1: Epoch-level aggregated train vs val loss
epochs = epoch_summary.index.values

fig, ax = plt.subplots(figsize=(9, 5))

if 'train_loss_mean' in epoch_summary.columns:
    m, s = epoch_summary['train_loss_mean'].values, epoch_summary['train_loss_std'].fillna(0).values
    ax.plot(epochs, m, marker='o', label='train_loss_mean')
    ax.fill_between(epochs, m - s, m + s, alpha=0.2)

if 'val_loss_mean' in epoch_summary.columns:
    m, s = epoch_summary['val_loss_mean'].values, epoch_summary['val_loss_std'].fillna(0).values
    ax.plot(epochs, m, marker='s', label='val_loss_mean')
    ax.fill_between(epochs, m - s, m + s, alpha=0.2)

ax.set_xlabel('epoch')
ax.set_ylabel('loss')
ax.set_yscale('log')
ax.set_title('Epoch-level aggregated loss (mean ± std)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Step-level train_loss with smoothing + val_loss overlay
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(train_df['step'], train_df['train_loss'], color='C0', alpha=0.25, label='train_loss (raw)')
ax.plot(train_df['step'], train_df['train_loss_sma'], color='C0', linewidth=2, label='train_loss SMA')
ax.plot(train_df['step'], train_df['train_loss_ewm'], color='C1', linestyle='--', linewidth=2, label='train_loss EWM')

if not val_df.empty:
    ax.scatter(val_df['step'], val_df['val_loss'], color='C2', s=40, zorder=5, label='val_loss')

ax.set_xlabel('global step')
ax.set_ylabel('loss')
ax.set_title('Training loss (step-level) + validation loss')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: All numeric metrics as subplots (step-level)
skip = {'epoch', 'step', 'train_loss_sma', 'train_loss_ewm', 'train_full_mse_sma', 'train_full_mse_ewm'}
metric_bases = [c for c in raw.columns if c not in skip and raw[c].notna().any()]

n = len(metric_bases)
fig, axs = plt.subplots(n, 1, figsize=(12, 4 * n), sharex=True)
if n == 1:
    axs = [axs]
skip_step = 10
for ax, col in zip(axs, metric_bases):
    # Use the appropriate sub-dataframe
    if col in train_df.columns and train_df[col].notna().any():
        sub = train_df
    elif col in val_df.columns and val_df[col].notna().any():
        sub = val_df
    else:
        sub = raw_filled

    sub_clean = sub.dropna(subset=[col])
    ax.plot(sub_clean['step'][skip_step:], sub_clean[col][skip_step:], color='C0', alpha=0.35, label=col)

    # Overlay smoothed versions if available
    for suf, color, ls in [('_sma', 'C0', '-'), ('_ewm', 'C1', '--')]:
        sc = col + suf
        if sc in train_df.columns and train_df[sc].notna().any():
            ax.plot(train_df['step'][skip_step:], train_df[sc][skip_step:], color=color, linestyle=ls, linewidth=2, label=sc)
    #ax.set_yscale('log')
    ax.set_ylabel(col)
    ax.legend(loc='upper right')
    ax.grid(True)

axs[-1].set_xlabel('global step')
plt.suptitle('All metrics vs global step', y=1.002)
plt.tight_layout()
plt.show()